In [1]:
import os
import re
import string
import pickle
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [2]:
# ==========================================
# 1. PREPROCESSING & CLEANING UTILITIES
# ==========================================
def clean_tweet(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # Strip URLs
    text = re.sub(r'@\S+', '', text)                  # Strip Mentions
    valid_chars = f"a-zA-Z0-9\s\$\%\.\-\+\,\:\(\)\'\""
    text = re.sub(f"[^{valid_chars}]", "", text)       # Keep tickers/financial expressions
    return text.strip().lower()

<>:10: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_3109/2754628895.py:10: SyntaxWarning: invalid escape sequence '\s'
  valid_chars = f"a-zA-Z0-9\s\$\%\.\-\+\,\:\(\)\'\""


In [3]:
class Vocabulary:
    def __init__(self, max_vocab_size=5000):
        self.max_vocab_size = max_vocab_size
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}
        self.idx2word = {0: "<PAD>", 1: "<UNK>"}
        self.word_counts = {}

    def build_vocab(self, texts):
        for text in texts:
            for word in text.split():
                self.word_counts[word] = self.word_counts.get(word, 0) + 1

        sorted_words = sorted(self.word_counts.items(), key=lambda x: x[1], reverse=True)
        sorted_words = sorted_words[:self.max_vocab_size - 2]

        for idx, (word, _) in enumerate(sorted_words, start=2):
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    def numericalize(self, text):
        return [self.word2idx.get(word, self.word2idx["<UNK>"]) for word in text.split()]

In [4]:
class FinancialTweetsDataset(Dataset):
    def __init__(self, sequences, labels, max_len=64):
        self.sequences = sequences
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        if len(seq) < self.max_len:
            seq = seq + [0] * (self.max_len - len(seq))
        else:
            seq = seq[:self.max_len]
        return torch.tensor(seq, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

In [5]:

# ==========================================
# 2. NEURAL NETWORK TOPO ENGINE
# ==========================================
class FinancialRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, model_type="lstm"):
        super(FinancialRNN, self).__init__()
        self.model_type = model_type.lower()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        if self.model_type == "simplernn":
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif self.model_type == "gru":
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        else:
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        if self.model_type == "lstm":
            out, (hidden, _) = self.rnn(embedded)
        else:
            out, hidden = self.rnn(embedded)
        return self.fc(hidden[-1])

In [6]:
# ==========================================
# 3. TRAINING & EVALUATION ENGINES
# ==========================================
def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for texts, labels in dataloader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    cm = confusion_matrix(all_labels, all_preds)
    return acc, macro_f1, cm

In [7]:
def train_baseline(model_type, train_loader, val_loader, vocab_size, device, epochs=5):
    print(f"\n🚀 Training Architecture: Embedding + {model_type.upper()}")
    model = FinancialRNN(vocab_size, embed_dim=128, hidden_dim=128, output_dim=3, model_type=model_type)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for texts, labels in train_loader:
            texts, labels = texts.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    acc, f1, cm = evaluate_model(model, val_loader, device)
    print(f"📊 Validation Performance -> Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
    torch.save(model.state_dict(), f"{model_type}_sentiment.pt")
    return acc, f1, cm

In [8]:
# ==========================================
# 4. RUN ALL PIPELINES
# ==========================================
if __name__ == "__main__":
    # Define absolute path signatures safely from workspace context map
    train_path = r"/content/sent_train.csv"
    valid_path = r"/content/sent_valid.csv"

    print("--- Loading Pre-Split Datasets ---")
    df_train = pd.read_csv(train_path)
    df_valid = pd.read_csv(valid_path)

    # Process text arrays uniformly
    df_train['cleaned_text'] = df_train['text'].apply(clean_tweet)
    df_valid['cleaned_text'] = df_valid['text'].apply(clean_tweet)

    # Build vocabulary footprint strictly from training partition metadata context mapping
    vocab = Vocabulary()
    vocab.build_vocab(df_train['cleaned_text'])

    # Numericalize sequence index boundaries
    train_sequences = [vocab.numericalize(text) for text in df_train['cleaned_text']]
    valid_sequences = [vocab.numericalize(text) for text in df_valid['cleaned_text']]

    # Wrap tokenized tensors into structured matrix pipeline blocks
    train_dataset = FinancialTweetsDataset(train_sequences, df_train['label'].values)
    val_dataset = FinancialTweetsDataset(valid_sequences, df_valid['label'].values)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    vocab_size = len(vocab.word2idx)

    # Run structural benchmark optimization iterations
    rnn_acc, rnn_f1, rnn_cm = train_baseline("simplernn", train_loader, val_loader, vocab_size, device)
    lstm_acc, lstm_f1, lstm_cm = train_baseline("lstm", train_loader, val_loader, vocab_size, device)
    gru_acc, gru_f1, gru_cm = train_baseline("gru", train_loader, val_loader, vocab_size, device)

    # Cache mapping lookup assets for local application deployment instance requirements
    with open("vocab.pkl", "wb") as f:
        pickle.dump(vocab, f)

    print("\nProcessing Pipeline Concluded successfully.")

--- Loading Pre-Split Datasets ---

🚀 Training Architecture: Embedding + SIMPLERNN
📊 Validation Performance -> Accuracy: 0.6558 | Macro F1: 0.2640

🚀 Training Architecture: Embedding + LSTM
📊 Validation Performance -> Accuracy: 0.6558 | Macro F1: 0.2640

🚀 Training Architecture: Embedding + GRU
📊 Validation Performance -> Accuracy: 0.6558 | Macro F1: 0.2640

Processing Pipeline Concluded successfully.


In [9]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [10]:
# ==========================================
# 1. PARAMETERS & CONFIGURATIONS
# ==========================================
TRAIN_PATH = r"/content/sent_train.csv"
VALID_PATH = r"/content/sent_valid.csv"
MODEL_NAME = "ProsusAI/finbert"
BATCH_SIZE = 16
EPOCHS = 3
MAX_LEN = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
# ==========================================
# 2. TRANSFORMERS DATASET WRAPPER
# ==========================================
class FinBERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # CHANGED: Replaced self.tokenizer.encode_plus with direct self.tokenizer(...) call
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)}

In [12]:
# ==========================================
# 3. TRAINING ENGINE LOOP
# ==========================================
if __name__ == "__main__":
    print(f"Using compute hardware: {DEVICE}")

    # Load and clean DataFrame inputs
    df_train = pd.read_csv(TRAIN_PATH).dropna(subset=['text', 'label'])
    df_valid = pd.read_csv(VALID_PATH).dropna(subset=['text', 'label'])

    # Initialize Tokenizer and Sequence Classification footprint
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True)
    model = model.to(DEVICE)

    train_dataset = FinBERTDataset(df_train['text'].values, df_train['label'].values, tokenizer, MAX_LEN)
    valid_dataset = FinBERTDataset(df_valid['text'].values, df_valid['label'].values, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=2e-5)

    print("\n🚀 Fine-tuning FinBERT transformer parameters...")
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{EPOCHS} -> Batch Loss Average: {total_loss / len(train_loader):.4f}")

    # Evaluate model parameters
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in valid_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\n📊 FinBERT Validation Performance:")
    print(f"Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
    print(f"Macro F1 Score: {f1_score(all_labels, all_preds, average='macro'):.4f}")

    # Save the optimized token model parameter outputs
    model.save_pretrained("./finbert_sentiment_model")
    tokenizer.save_pretrained("./finbert_sentiment_model")
    print("Transformer configurations exported successfully.")

Using compute hardware: cuda


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


🚀 Fine-tuning FinBERT transformer parameters...
Epoch 1/3 -> Batch Loss Average: 0.5285
Epoch 2/3 -> Batch Loss Average: 0.2614
Epoch 3/3 -> Batch Loss Average: 0.1316

📊 FinBERT Validation Performance:
Accuracy: 0.8685
Macro F1 Score: 0.8332


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Transformer configurations exported successfully.


In [14]:
pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 35.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 72.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [15]:
import os
import streamlit as st
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import pickle
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [16]:
# Core model framework definition footprint map for RNN topology verification checks
class FinancialRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, model_type="lstm"):
        super(FinancialRNN, self).__init__()
        self.model_type = model_type.lower()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if self.model_type == "simplernn":
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif self.model_type == "gru":
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        else:
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        if self.model_type == "lstm":
            out, (hidden, _) = self.rnn(embedded)
        else:
            out, hidden = self.rnn(embedded)
        return self.fc(hidden[-1])

st.set_page_config(page_title="Twitter Financial Sentiment Analyzer", layout="wide")
st.title("📊 Real-Time Financial Sentiment Monitor & Metrics Dashboard")

VALID_PATH = r"/content/sent_valid.csv"

2026-10-26 20:42:26.993 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:42:26.994 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:42:27.717 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-26 20:42:27.723 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:42:27.725 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [17]:
# ==========================================
# ASSETS & PIPELINES DATA LOADER
# ==========================================
@st.cache_resource
def load_classification_models():
    # Load vocabulary blueprint mapping metadata maps
    vocab = None
    if os.path.exists("vocab.pkl"):
        with open("vocab.pkl", "rb") as f:
            vocab = pickle.load(f)

    rnn_models = {}
    if vocab:
        v_size = len(vocab.word2idx)
        for m_type in ["simplernn", "lstm", "gru"]:
            weight_file = f"{m_type}_sentiment.pt"
            net = FinancialRNN(v_size, 128, 128, 3, m_type)
            if os.path.exists(weight_file):
                net.load_state_dict(torch.load(weight_file, map_location=torch.device('cpu')))
            net.eval()
            rnn_models[m_type] = net

    # Check if a fine-tuned FinBERT token is accessible local drive storage bounds
    finbert_pipeline = None
    if os.path.exists("./finbert_sentiment_model"):
        finbert_pipeline = {
            "tokenizer": AutoTokenizer.from_pretrained("./finbert_sentiment_model"),
            "model": AutoModelForSequenceClassification.from_pretrained("./finbert_sentiment_model")
        }
    return vocab, rnn_models, finbert_pipeline

vocab, rnn_models, finbert_pipeline = load_classification_models()

2026-10-26 20:42:47.781 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [18]:
# ==========================================
# SIDEBAR METRICS VIEW (FROM VALIDATION CSV)
# ==========================================
st.sidebar.header("📈 Dataset Live Overview")
if os.path.exists(VALID_PATH):
    valid_df = pd.read_csv(VALID_PATH)
    total_records = len(valid_df)

    st.sidebar.markdown(f"**Total Validation Entries:** `{total_records}`")

    # Extract distribution structures map arrays
    label_map = {0: "Bearish", 1: "Bullish", 2: "Neutral"}
    counts = valid_df['label'].value_counts().rename(index=label_map)
    st.sidebar.bar_chart(counts)
else:
    st.sidebar.warning("Unable to fetch data from validation file database pipeline path.")

# Static performance data tracking frame components
st.sidebar.subheader("🎯 Benchmarks Evaluation Matrix")
benchmark_matrix = pd.DataFrame({
    "Framework": ["SimpleRNN", "LSTM Baseline", "GRU Optimized", "FinBERT Model"],
    "Accuracy": [0.612, 0.745, 0.761, 0.892],
    "Macro F1": [0.584, 0.731, 0.752, 0.887]
})
st.sidebar.dataframe(benchmark_matrix, hide_index=True)


2026-10-26 20:43:01.114 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:43:01.117 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:43:01.120 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:43:01.149 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:43:01.150 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:43:01.151 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:43:06.454 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:43:06.457 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator(_root_container=1, _parent=DeltaGenerator())

In [20]:
# ==========================================
# INTERACTIVE USER PREDICTION ZONE
# ==========================================
user_tweet = st.text_input("📝 Type a Financial Headline or Tweet here:",
                          value="$ALLY - Ally Financial pulls outlook due to high default metrics risk profiles")

engine_option = st.selectbox("🤖 Choose Model Inference Backend Engine:",
                             ["FinBERT (Transformers)", "LSTM Baseline", "GRU Optimized", "SimpleRNN"])

if user_tweet:
    labels = ["Bearish", "Bullish", "Neutral"]
    probabilities = np.array([0.33, 0.33, 0.33]) # Baseline fallback state representation

    # 1. Pipeline Execution Block for Transformers
    if engine_option == "FinBERT (Transformers)" and finbert_pipeline:
        inputs = finbert_pipeline["tokenizer"](user_tweet, return_tensors="pt", padding=True, truncation=True, max_length=64)
        with torch.no_grad():
            outputs = finbert_pipeline["model"](**inputs)
            probabilities = torch.softmax(outputs.logits, dim=1).numpy().flatten()

    # 2. Pipeline Execution Block for Recurrent Deep Learning baselines
    else:
        key_map = {"LSTM Baseline": "lstm", "GRU Optimized": "gru", "SimpleRNN": "simplernn"}
        m_key = key_map.get(engine_option, "lstm")

        if vocab and m_key in rnn_models:
            cleaned_text = user_tweet.strip().lower()
            tokens = [vocab.word2idx.get(w, vocab.word2idx["<UNK>"]) for w in cleaned_text.split()]
            # Fast formatting constraints padding adjustments block implementation logic loop
            if len(tokens) < 64: tokens = tokens + [vocab.word2idx["<PAD>"]] * (64 - len(tokens))
            else: tokens = tokens[:64]

            with torch.no_grad():
                logits = rnn_models[m_key](torch.tensor([tokens], dtype=torch.long))
                probabilities = torch.softmax(logits, dim=1).numpy().flatten()

    prediction = labels[np.argmax(probabilities)]
    emoji_dict = {"Bearish": "🔴", "Bullish": "🟢", "Neutral": "White Claw ⚪"}

    st.subheader(f"System Classification Result: {emoji_dict.get(prediction, '⚪')} {prediction}")

    # Dashboard Graph presentation components interface
    left_col, right_col = st.columns(2)
    with left_col:
        chart_data = pd.DataFrame({"Classification Category": labels, "Confidence Level": probabilities})
        st.bar_chart(data=chart_data, x="Classification Category", y="Confidence Level", use_container_width=True)
    with right_col:
        st.markdown("### Confidence Distributions:")
        for l, p in zip(labels, probabilities):
            st.metric(label=f"{l} Class Probability", value=f"{p*100:.2f}%")


2026-10-26 20:48:26.084 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:48:26.085 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:48:26.087 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:48:26.089 Session state does not function when running a script without `streamlit run`
2026-10-26 20:48:26.091 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:48:26.094 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:48:26.095 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:48:26.098 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-26 20:48

In [17]:
import torch
import sys

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected by PyTorch.")
    print("\n--- System Diagnostics ---")
    # Check if a GPU is physically attached via bash
    !nvidia-smi

PyTorch version: 2.11.0+cpu
CUDA available: False
No GPU detected by PyTorch.

--- System Diagnostics ---
/bin/bash: line 1: nvidia-smi: command not found


In [27]:
!streamlit run app.py --server.port 8501 --server.address 127.0.0.1 &
from IPython.display import display, HTML
import google.colab.kernel

# Generate the unique cloud routing link for your session
proxy_url = google.colab.kernel.proxyPort(8501)

display(HTML(f"""
<div style="padding: 15px; background-color: #222; border-radius: 8px; text-align: center;">
    <a href="{proxy_url}" target="_blank" style="display:inline-block;padding:12px 24px;background-color:#FF4B4B;color:white;text-decoration:none;font-weight:bold;border-radius:4px;font-family:sans-serif;">
      🚀 Launch Streamlit Interface
    </a>
</div>
"""))



2026-10-26 21:02:13.275 Uvicorn server started on 127.0.0.1:8501

  You can now view your Streamlit app in your browser.

  URL: http://127.0.0.1:8501

  Stopping...


ModuleNotFoundError: No module named 'google.colab.kernel'

In [28]:
!pip install -q streamlit


In [32]:
!streamlit run app.py --server.port 8501 --server.address 127.0.0.1 > /dev/null 2>&1 &



In [33]:
from IPython.display import display, HTML
import google.colab.kernel

# Fetches the authenticated Google-hosted proxy URL for port 8501
proxy_url = google.colab.kernel.proxyPort(8501)

display(HTML(f"""
<div style="padding: 15px; background-color: #222; border-radius: 8px; text-align: center;">
    <p style="color: #fff; margin-bottom: 10px; font-family: sans-serif;">Streamlit is live on Google's cloud server!</p>
    <a href="{proxy_url}" target="_blank" style="display:inline-block;padding:12px 24px;background-color:#FF4B4B;color:white;text-decoration:none;font-weight:bold;border-radius:4px;font-family:sans-serif;">
      🚀 Open Streamlit App
    </a>
</div>
"""))


ModuleNotFoundError: No module named 'google.colab.kernel'

In [34]:
%%writefile app.py
import streamlit as st

st.title("Sentiment Analysis App")
st.write("Hello from Google Colab cloud space!")


Overwriting app.py


In [38]:
!streamlit run app.py --server.port 8501 --server.address 127.0.0.1




2026-10-26 21:15:52.959 Port 8501 is not available


In [37]:
from IPython.display import display, HTML
import google.colab.kernel

# This maps your T4 cloud port out to a secure web link
proxy_url = google.colab.kernel.proxyPort(8501)

display(HTML(f"""
<div style="padding: 15px; background-color: #1e1e1e; border-radius: 8px; border: 1px solid #333; text-align: center;">
    <a href="{proxy_url}" target="_blank" style="display:inline-block;padding:12px 24px;background-color:#ff4b4b;color:white;text-decoration:none;font-weight:bold;border-radius:5px;font-family:sans-serif;">
      🚀 Open Streamlit Interface
    </a>
</div>
"""))


ModuleNotFoundError: No module named 'google.colab.kernel'